# 🥈 Step 2 — Transform & Clean to Silver

**Purpose:** Read raw Bronze tables, apply data-quality transformations, and persist
the results as cleaned Delta tables in the `silver` schema.

### Transformations Applied

| Bronze Table | Silver Table | Transformations |
|---|---|---|
| `raw_orders` | `cleaned_orders` | Cast timestamps, deduplicate on `order_id`, drop rows missing `order_id` or `customer_id` |
| `raw_order_items` | `cleaned_order_items` | Cast `price` & `freight_value` to float, deduplicate on (`order_id`, `order_item_id`) |

> **Prerequisites:** Run `01_ingest_to_bronze` first so the Bronze tables exist.

---

In [0]:
# ---------------------------------------------------------------------------
# Imports
# ---------------------------------------------------------------------------
# col          — reference columns by name in DataFrame transformations
# to_timestamp — cast string dates from Bronze into proper TimestampType
from pyspark.sql.functions import col, to_timestamp

In [0]:
# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
catalog = "medallion_trial"  # Must match the catalog used in the Bronze notebook

---
## 📦 Transform 1 — Orders Table

**Source:** `bronze.raw_orders` → **Target:** `silver.cleaned_orders`

Transformations:
- Cast `order_purchase_timestamp` and `order_delivered_customer_date` from string → TimestampType
- Remove duplicate orders (by `order_id`)
- Drop rows where `order_id` or `customer_id` is null (data-quality gate)

In [0]:
print("Starting Silver Layer Transformations...")

# ── 1. Read the raw orders from Bronze ────────────────────────────────────
df_raw_orders = spark.table(f"{catalog}.bronze.raw_orders")

# ── 2. Apply cleaning transformations ─────────────────────────────────────
df_silver_orders = (
    df_raw_orders
    # Cast string timestamps → proper TimestampType for downstream analytics
    .withColumn("order_purchase_timestamp",
                to_timestamp(col("order_purchase_timestamp")))
    .withColumn("order_delivered_customer_date",
                to_timestamp(col("order_delivered_customer_date")))
    # Remove exact-duplicate orders
    .dropDuplicates(["order_id"])
    # Data-quality gate: discard rows missing critical identifiers
    .dropna(subset=["order_id", "customer_id"])
)

# ── 3. Write to Silver as a managed Delta table ───────────────────────────
(
    df_silver_orders.write.format("delta")
    .mode("overwrite")
    .saveAsTable(f"{catalog}.silver.cleaned_orders")
)

print("✅ Cleaned Orders saved to Silver!")

---
## 📦 Transform 2 — Order Items Table

**Source:** `bronze.raw_order_items` → **Target:** `silver.cleaned_order_items`

Transformations:
- Cast `price` and `freight_value` from string → FloatType for numeric operations
- Deduplicate on the composite key (`order_id`, `order_item_id`)

In [0]:
# ── 1. Read the raw order items from Bronze ──────────────────────────────
df_raw_items = spark.table(f"{catalog}.bronze.raw_order_items")

# ── 2. Apply cleaning transformations ─────────────────────────────────────
df_silver_items = (
    df_raw_items
    # Cast monetary columns from string → float so they can be summed in Gold
    .withColumn("price", col("price").cast("float"))
    .withColumn("freight_value", col("freight_value").cast("float"))
    # Deduplicate on the natural composite key
    .dropDuplicates(["order_id", "order_item_id"])
)

# ── 3. Write to Silver as a managed Delta table ───────────────────────────
(
    df_silver_items.write.format("delta")
    .mode("overwrite")
    .saveAsTable(f"{catalog}.silver.cleaned_order_items")
)

print("✅ Cleaned Order Items saved to Silver!")
print("\n🎉 Silver layer transformations complete!")